![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 04</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Tuning Unity Catalog managed Tables for Analytical Queries</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">See how Predictive Optimization and automatic liquid clustering keep UC managed tables fast for analytical reads, mostly without manual work.</div>
</div>
</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.

In [0]:
%run ./Includes/Classroom-Setup-04

## What this lesson covers

Every table in this course is a **UC managed table**, the storage layer that powers analytical workloads on Databricks. A few things govern how fast analytical scans run: how data files are laid out, how rows are clustered, and whether the query planner has good statistics.

The big shift in recent years is that **most of this tuning now happens automatically.** **Predictive Optimization** watches how your tables are written and queried and runs the right maintenance for you, on serverless compute, in the background. You no longer schedule these operations by hand for most tables.

In this lesson you will:

1. Inspect a table's physical layout with `DESCRIBE EXTENDED`.
2. See what **Predictive Optimization** does for you and confirm it's enabled.
3. Use **automatic liquid clustering** (`CLUSTER BY AUTO`), the modern replacement for partitioning and Z-ordering.
4. Look at the manual commands (`OPTIMIZE`, `ANALYZE`, `VACUUM`) that Predictive Optimization runs automatically, and learn when you'd still reach for them yourself.
5. Make a couple of changes to a table and read its version history.
6. Explore lineage and history in Catalog Explorer.

## 1. Where the table lives

`DESCRIBE EXTENDED` returns columns, data types, storage location, and provider. You ran this in Lesson 02. The output has three sections separated by blank rows:

<div style="max-width: 880px; margin: 20px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 14px; align-items: stretch;">
    <!-- Part 1: Columns -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px;">1</span>
        Column Schema
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1;">
        <div style="font-size: 15px; color: #3d5a65; margin-bottom: 10px;"><strong>Names + data types</strong></div>
        <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 10px; font-family: monospace; font-size: 14px; line-height: 1.7; color: #1B3139;">
          customer_id&nbsp;&nbsp;&nbsp;&nbsp;string<br/>
          customer_name&nbsp;&nbsp;string<br/>
          city&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;string<br/>
          postal_code&nbsp;&nbsp;&nbsp;&nbsp;int<br/>
          <span style="color: #5E7077;">... 6 more columns</span>
        </div>
      </div>
    </div>
    <!-- Part 2: Delta Statistics -->
    <div style="flex: 1; border: 2px solid #FF8C42; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #FF8C42; color: #fff; padding: 10px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px;">2</span>
        Delta Statistics
      </div>
      <div style="padding: 14px; background: #FFFAF6; flex: 1;">
        <div style="font-size: 15px; color: #3d5a65; margin-bottom: 10px;"><strong>Min/max stats for file skipping</strong></div>
        <div style="display: flex; flex-direction: column; gap: 6px;">
          <div style="background: #fff; border: 1px solid #E8D5C4; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Method</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">first-32</span>
          </div>
          <div style="background: #fff; border: 1px solid #E8D5C4; border-radius: 6px; padding: 8px 10px; font-size: 14px; color: #3d5a65; line-height: 1.5;">
            <strong>Tracked columns</strong><br/>
            <span style="font-family: monospace;">customer_id, customer_name, email, city, state, postal_code, ...</span>
          </div>
        </div>
      </div>
    </div>
    <!-- Part 3: Table Info -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px;">3</span>
        Table Metadata
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1;">
        <div style="font-size: 15px; color: #3d5a65; margin-bottom: 10px;"><strong>Location, format, size</strong></div>
        <div style="display: flex; flex-direction: column; gap: 6px;">
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Provider</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">delta</span>
          </div>
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Type</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">MANAGED</span>
          </div>
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Location</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">s3://...</span>
          </div>
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Statistics</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">5,000 rows</span>
          </div>
        </div>
      </div>
    </div>
  </div>
</div>

In [0]:
%sql
DESCRIBE EXTENDED customers;

In the **Detailed Table Information** section near the bottom, the `Location` row points at a path under your catalog's managed storage. That folder holds the underlying Parquet files plus a `_delta_log` directory that records every write.

Because `customers` is a **Unity Catalog managed table**, it's exactly the kind of table Predictive Optimization looks after, which is what the next section is about.

## 2. Predictive Optimization: tuning that runs itself

**Predictive Optimization (PO)** is an AI-driven service that maintains your Unity Catalog managed tables for you. It continuously analyzes how each table is written and queried, then runs the right maintenance automatically on serverless compute, without you scheduling anything.

<div style="width: 100%; margin: 16px 0;">
<svg viewBox="0 0 720 300" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <text x="360" y="30" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">WHAT PREDICTIVE OPTIMIZATION DOES FOR YOU</text>
  <rect x="270" y="120" width="180" height="70" rx="10" fill="#1B5162"/>
  <text x="360" y="150" text-anchor="middle" fill="#FFFFFF" font-size="16" font-weight="700">Predictive</text>
  <text x="360" y="172" text-anchor="middle" fill="#FFFFFF" font-size="16" font-weight="700">Optimization</text>
  <rect x="30" y="70" width="190" height="56" rx="8" fill="#F9F7F4" stroke="#00A972" stroke-width="1.6"/>
  <text x="125" y="93" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Compact small files</text>
  <text x="125" y="113" text-anchor="middle" fill="#5E7077" font-size="15">runs OPTIMIZE for you</text>
  <rect x="30" y="135" width="190" height="56" rx="8" fill="#F9F7F4" stroke="#00A972" stroke-width="1.6"/>
  <text x="125" y="158" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Cluster data layout</text>
  <text x="125" y="178" text-anchor="middle" fill="#5E7077" font-size="15">picks clustering keys</text>
  <rect x="500" y="70" width="190" height="56" rx="8" fill="#F9F7F4" stroke="#00A972" stroke-width="1.6"/>
  <text x="595" y="93" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Collect statistics</text>
  <text x="595" y="113" text-anchor="middle" fill="#5E7077" font-size="15">on write, no ANALYZE</text>
  <rect x="500" y="135" width="190" height="56" rx="8" fill="#F9F7F4" stroke="#00A972" stroke-width="1.6"/>
  <text x="595" y="158" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Vacuum old files</text>
  <text x="595" y="178" text-anchor="middle" fill="#5E7077" font-size="15">reclaims storage</text>
  <line x1="220" y1="98" x2="270" y2="145" stroke="#B5BFC4" stroke-width="1.6"/>
  <line x1="220" y1="163" x2="270" y2="158" stroke="#B5BFC4" stroke-width="1.6"/>
  <line x1="450" y1="145" x2="500" y2="98" stroke="#B5BFC4" stroke-width="1.6"/>
  <line x1="450" y1="158" x2="500" y2="163" stroke="#B5BFC4" stroke-width="1.6"/>
  <text x="360" y="240" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">Runs automatically on serverless compute, based on how each table is used.</text>
  <text x="360" y="262" text-anchor="middle" fill="#5E7077" font-size="15" font-style="italic">No schedules to set up and no maintenance jobs to manage.</text>
</svg>
</div>

With PO enabled, Databricks automatically:

- **Compacts small files** into larger ones (the work the `OPTIMIZE` command does), so scans open fewer files.
- **Clusters data layout** when automatic liquid clustering is on (next section), choosing and adapting the clustering keys from your query patterns.
- **Collects column statistics** as data is written, so the query planner makes better decisions, no separate `ANALYZE` pass needed.
- **Vacuums** unreferenced old files to reclaim storage.

PO is **enabled by default** for new accounts and tables and has been rolling out across existing accounts, so on a current workspace it is typically already on. It's the recommended default for all Unity Catalog managed tables.

### Confirm it's enabled

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 12px 0; font-size: 15px; color: #333; line-height: 1.6;">
  <strong style="color: #1565c0; font-size: 16px;">Try this in Catalog Explorer</strong>
  <ol style="margin: 8px 0 0 0; padding-left: 20px;">
    <li>In the left sidebar, click <strong>Catalog</strong>.</li>
    <li>Navigate to your catalog &#x2192; <code style="font-size: 15px;">vintage_audio</code> schema &#x2192; <code style="font-size: 15px;">customers</code> table.</li>
    <li>Open the table's <strong>Details</strong> tab and look for the <strong>Predictive Optimization</strong> field. It shows whether PO is enabled and whether the setting is inherited from the catalog or schema.</li>
  </ol>
</div>

If it shows as inherited and enabled, you don't need to do anything. PO is already maintaining this table. (Enabling or disabling PO is an account/catalog/schema-level admin action, so it's not something you toggle per notebook.)

## 3. Automatic liquid clustering: the modern layout lever

To make filtered scans fast, the rows you query together should sit in the same files, so the engine can **skip** files that can't match. Historically you controlled this with partitioning or `OPTIMIZE ZORDER BY`. Today the recommended approach is **liquid clustering**, and specifically **automatic liquid clustering**.

<div style="width: 100%; margin: 16px 0;">
<svg viewBox="0 0 760 470" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <text x="380" y="30" text-anchor="middle" fill="#1B5162" font-size="19" font-weight="700">HOW CLUSTERING MAKES SCANS FAST</text>
  <text x="380" y="54" text-anchor="middle" fill="#5E7077" font-size="15" font-family="monospace">SELECT * FROM customers WHERE state = 'CA';</text>
  <rect x="20" y="74" width="720" height="180" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
  <text x="40" y="102" fill="#FE3722" font-size="15" font-weight="700">UNCLUSTERED &#8212; CA rows scattered, every file scanned</text>
  <rect x="40" y="118" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.5"/>
  <text x="120" y="142" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">file_001</text>
  <text x="120" y="162" text-anchor="middle" fill="#5E7077" font-size="14">AK, CA, FL, NY</text>
  <text x="120" y="178" text-anchor="middle" fill="#FE3722" font-size="14" font-weight="700">SCANNED</text>
  <rect x="212" y="118" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.5"/>
  <text x="292" y="142" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">file_002</text>
  <text x="292" y="162" text-anchor="middle" fill="#5E7077" font-size="14">AZ, CA, GA, OH</text>
  <text x="292" y="178" text-anchor="middle" fill="#FE3722" font-size="14" font-weight="700">SCANNED</text>
  <rect x="384" y="118" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.5"/>
  <text x="464" y="142" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">file_003</text>
  <text x="464" y="162" text-anchor="middle" fill="#5E7077" font-size="14">CA, IL, MA, TX</text>
  <text x="464" y="178" text-anchor="middle" fill="#FE3722" font-size="14" font-weight="700">SCANNED</text>
  <rect x="556" y="118" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#FE3722" stroke-width="1.5"/>
  <text x="636" y="142" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">file_004</text>
  <text x="636" y="162" text-anchor="middle" fill="#5E7077" font-size="14">CA, NV, OR, WA</text>
  <text x="636" y="178" text-anchor="middle" fill="#FE3722" font-size="14" font-weight="700">SCANNED</text>
  <text x="40" y="222" fill="#0b2026" font-size="15" font-weight="700">All 4 files scanned.</text>
  <text x="40" y="242" fill="#5E7077" font-size="14">CA rows are mixed into every file, so the engine must read them all.</text>
  <rect x="20" y="270" width="720" height="184" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
  <text x="40" y="298" fill="#00A972" font-size="15" font-weight="700">CLUSTERED BY state &#8212; CA rows together, other files skipped</text>
  <rect x="40" y="314" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#00A972" stroke-width="1.5"/>
  <text x="120" y="338" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">file_001</text>
  <text x="120" y="358" text-anchor="middle" fill="#5E7077" font-size="14">CA, CA, CA, CA</text>
  <text x="120" y="374" text-anchor="middle" fill="#00A972" font-size="14" font-weight="700">SCANNED</text>
  <rect x="212" y="314" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#B5BFC4" stroke-width="1.2" stroke-dasharray="4,3"/>
  <text x="292" y="338" text-anchor="middle" fill="#5E7077" font-size="15" font-weight="700">file_002</text>
  <text x="292" y="358" text-anchor="middle" fill="#B5BFC4" font-size="14">FL, GA, HI, IA</text>
  <text x="292" y="374" text-anchor="middle" fill="#5E7077" font-size="14" font-weight="700">SKIPPED</text>
  <rect x="384" y="314" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#B5BFC4" stroke-width="1.2" stroke-dasharray="4,3"/>
  <text x="464" y="338" text-anchor="middle" fill="#5E7077" font-size="15" font-weight="700">file_003</text>
  <text x="464" y="358" text-anchor="middle" fill="#B5BFC4" font-size="14">IL, IN, KS, KY</text>
  <text x="464" y="374" text-anchor="middle" fill="#5E7077" font-size="14" font-weight="700">SKIPPED</text>
  <rect x="556" y="314" width="160" height="64" rx="5" fill="#FFFFFF" stroke="#B5BFC4" stroke-width="1.2" stroke-dasharray="4,3"/>
  <text x="636" y="338" text-anchor="middle" fill="#5E7077" font-size="15" font-weight="700">file_004</text>
  <text x="636" y="358" text-anchor="middle" fill="#B5BFC4" font-size="14">NY, OH, PA, TX</text>
  <text x="636" y="374" text-anchor="middle" fill="#5E7077" font-size="14" font-weight="700">SKIPPED</text>
  <text x="40" y="418" fill="#0b2026" font-size="15" font-weight="700">Only 1 of 4 files scanned.</text>
  <text x="40" y="438" fill="#5E7077" font-size="14">Rows are co-located by state, so file-skipping reads only what it must.</text>
</svg>
</div>

With **`CLUSTER BY AUTO`**, Predictive Optimization analyzes how the table is queried, **chooses the clustering keys for you**, and adapts them over time as usage changes. Liquid clustering replaces both partitioning and Z-ordering for the large majority of tables; it's flexible (you can change keys without rewriting the whole table) and it doesn't require you to guess the right columns up front.

### Turn on automatic liquid clustering

Enabling it is a single statement. Predictive Optimization then selects the clustering keys and clusters the data in the background, no key list required from you.

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 14px; align-items: stretch;">
    <!-- Option 1: AUTO -->
    <div style="flex: 1; border: 2px solid #00A972; border-radius: 10px; overflow: hidden;">
      <div style="background: #00A972; color: #fff; padding: 10px 14px; text-align: center;">
        <div style="font-size: 15px; font-weight: 700; font-family: monospace;">CLUSTER BY AUTO</div>
        <div style="font-size: 14px; opacity: 0.85;">Recommended default</div>
      </div>
      <div style="padding: 14px; background: #F0FAF5; font-size: 15px; color: #3d5a65;">
        <div style="display: flex; flex-direction: column; gap: 6px;">
          <div style="display: flex; align-items: center; gap: 8px;"><span style="color: #00A972; font-weight: 700;">&#10003;</span> PO picks clustering keys from query patterns</div>
          <div style="display: flex; align-items: center; gap: 8px;"><span style="color: #00A972; font-weight: 700;">&#10003;</span> Adapts over time as usage changes</div>
          <div style="display: flex; align-items: center; gap: 8px;"><span style="color: #00A972; font-weight: 700;">&#10003;</span> Best when you're unsure which columns matter most</div>
        </div>
      </div>
    </div>
    <!-- Option 2: Manual keys -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; text-align: center;">
        <div style="font-size: 15px; font-weight: 700; font-family: monospace;">CLUSTER BY (col1, col2)</div>
        <div style="font-size: 14px; opacity: 0.85;">When you know the columns</div>
      </div>
      <div style="padding: 14px; background: #f8fafc; font-size: 15px; color: #3d5a65;">
        <div style="display: flex; flex-direction: column; gap: 6px;">
          <div style="display: flex; align-items: center; gap: 8px;"><span style="color: #1B5162; font-weight: 700;">&#10003;</span> You specify exact clustering keys</div>
          <div style="display: flex; align-items: center; gap: 8px;"><span style="color: #1B5162; font-weight: 700;">&#10003;</span> Keys stay fixed until you change them</div>
          <div style="display: flex; align-items: center; gap: 8px;"><span style="color: #1B5162; font-weight: 700;">&#10003;</span> Best when query patterns are well known</div>
        </div>
      </div>
    </div>
  </div>
  <div style="font-size: 15px; color: #5E7077; margin-top: 10px; text-align: center;">Both replace partitioning and Z-ordering. No rewrite needed to change keys later.</div>
</div>

In [0]:
%sql
ALTER TABLE customers CLUSTER BY AUTO;

That's the whole setup. From here, PO decides which columns to cluster on based on how `customers` is actually queried, and keeps the layout current as those patterns change.

A few notes for real-world use:

- **Manual keys are still available.** If you already know the right columns, you can pin them with `CLUSTER BY (col1, col2)` instead of `AUTO`. Automatic is the better default when you're unsure or the workload shifts over time.
- **Liquid clustering replaces partitioning and Z-ordering.** You don't combine them; a clustered table doesn't use `PARTITIONED BY` or `ZORDER BY`.
- **Z-order still exists** and isn't wrong, but it's no longer what Databricks steers you toward for new tables. Reach for liquid clustering first.

> **A note on scale.** On this 5,000-row `customers` table you won't see a runtime difference: the data fits in a single file, so there's nothing to skip. On a production table with hundreds of millions of rows across many files, clustering routinely cuts scan time many times over. The technique scales; the demo dataset is just small.

## 4. The manual commands underneath

Predictive Optimization runs three operations for you. It's still worth seeing them directly once, both to understand what PO is doing and because you may need them when PO isn't available (for example, on tables that aren't UC managed, or when you want to force the work immediately).

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 14px;">
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">OPTIMIZE</div>
      <div style="padding: 14px; background: #f8fafc; font-size: 15px; color: #3d5a65; line-height: 1.5; text-align: center;">
        <div style="font-weight: 700; color: #1B3139; margin-bottom: 4px;">Compact files</div>
        <div>Merges small files into fewer large ones. Fewer files = faster scans.</div>
        <div style="margin-top: 8px; font-size: 14px; color: #00A972; font-weight: 600;">PO does this automatically</div>
      </div>
    </div>
    <div style="flex: 1; border: 2px solid #FF8C42; border-radius: 10px; overflow: hidden;">
      <div style="background: #FF8C42; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">ANALYZE</div>
      <div style="padding: 14px; background: #FFFAF6; font-size: 15px; color: #3d5a65; line-height: 1.5; text-align: center;">
        <div style="font-weight: 700; color: #1B3139; margin-bottom: 4px;">Collect statistics</div>
        <div>Gives the query planner row counts, min/max, and distinct counts per column.</div>
        <div style="margin-top: 8px; font-size: 14px; color: #00A972; font-weight: 600;">PO does this on write</div>
      </div>
    </div>
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 16px; font-weight: 700; text-align: center;">VACUUM</div>
      <div style="padding: 14px; background: #f8fafc; font-size: 15px; color: #3d5a65; line-height: 1.5; text-align: center;">
        <div style="font-weight: 700; color: #1B3139; margin-bottom: 4px;">Reclaim storage</div>
        <div>Deletes old files past the retention window (default 7 days).</div>
        <div style="margin-top: 8px; font-size: 14px; color: #00A972; font-weight: 600;">PO does this automatically</div>
      </div>
    </div>
  </div>
</div>

### OPTIMIZE - compact files now

`OPTIMIZE` compacts small files into larger ones. This is the file-compaction work PO performs automatically; running it by hand simply does it on demand.

In [0]:
%sql
OPTIMIZE customers;

The result reports the compaction metrics. On a small table the impact is modest; on a heavily-written production table it can cut scan time substantially. With PO enabled you normally never run this yourself.

### ANALYZE - statistics for the planner

The query planner uses column statistics (row counts, min/max, distinct counts) to choose good plans, especially for joins. With Predictive Optimization, **statistics are collected automatically as data is written**, so you rarely run `ANALYZE` yourself. You can still trigger it explicitly:

In [0]:
%sql
ANALYZE TABLE customers COMPUTE STATISTICS FOR ALL COLUMNS;

The value here is concrete: better statistics mean the planner stops guessing about how many rows a filter or join will produce, which leads to faster plans. Confirm the stats landed by describing a single column:

In [0]:
%sql
DESCRIBE EXTENDED customers state;

You should see `min`, `max`, `num_nulls`, `distinct_count`, and `avg_col_len` for the `state` column. Under PO, this kind of statistic is kept current for you in the background.

### VACUUM - reclaim storage from old files

When a table is rewritten, the old files aren't deleted right away; they're retained (default 7 days) so time-travel queries against recent versions still work. `VACUUM` deletes files past that retention window. **Vacuuming is also part of Predictive Optimization**, which removes unreferenced files for you on managed tables. Preview what would be removed with `DRY RUN`:

In [0]:
%sql
VACUUM customers DRY RUN;

On this freshly-created table the list is empty (no files are old enough yet). With PO handling vacuum automatically, manual `VACUUM` is mainly for tables PO doesn't manage. Avoid `VACUUM ... RETAIN 0 HOURS`: it deletes files immediately and breaks time travel.

## 5. Version history: actually change the table, then read it

Every write to a UC managed table is recorded as a new **version** in its transaction log. To see versions that show real change, let's make two small edits: insert a row, then delete it. (We add a clearly fake customer so the table is left exactly as it started.)

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="font-size: 15px; font-weight: 700; color: #1B5162; margin-bottom: 8px;">What DESCRIBE HISTORY will show after these two edits:</div>
  <div style="display: flex; gap: 0; align-items: center;">
    <div style="flex: 1; background: #f8fafc; border: 2px solid #DCE0E2; border-radius: 10px; padding: 10px; text-align: center;">
      <div style="font-size: 14px; font-weight: 700; color: #5E7077;">Version 0</div>
      <div style="font-size: 14px; font-weight: 700; color: #1B3139; margin-top: 4px;">CREATE TABLE</div>
      <div style="font-size: 14px; color: #5E7077;">Setup script</div>
    </div>
    <div style="flex: 0 0 20px; text-align: center; font-size: 18px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1.2; background: #f8fafc; border: 2px solid #DCE0E2; border-radius: 10px; padding: 10px; text-align: center;">
      <div style="font-size: 14px; font-weight: 700; color: #5E7077;">Versions 1-3</div>
      <div style="font-size: 14px; font-weight: 700; color: #1B3139; margin-top: 4px;">CLUSTER BY AUTO</div>
      <div style="font-size: 14px; color: #5E7077;">Protocol upgrade + backfill + cluster</div>
    </div>
    <div style="flex: 0 0 20px; text-align: center; font-size: 18px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1; background: #EDF5F7; border: 2px solid #1B5162; border-radius: 10px; padding: 10px; text-align: center;">
      <div style="font-size: 14px; font-weight: 700; color: #1B5162;">Version 4</div>
      <div style="font-size: 14px; font-weight: 700; color: #1B3139; margin-top: 4px;">WRITE</div>
      <div style="font-size: 14px; color: #1B5162;">INSERT one row &#x2193;</div>
    </div>
    <div style="flex: 0 0 20px; text-align: center; font-size: 18px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1; background: #EDF5F7; border: 2px solid #1B5162; border-radius: 10px; padding: 10px; text-align: center;">
      <div style="font-size: 14px; font-weight: 700; color: #1B5162;">Version 5</div>
      <div style="font-size: 14px; font-weight: 700; color: #1B3139; margin-top: 4px;">DELETE</div>
      <div style="font-size: 14px; color: #1B5162;">Remove that row &#x2193;</div>
    </div>
  </div>
</div>

In [0]:
%sql
-- Version N+1: insert one row
INSERT INTO customers (customer_id, customer_name, email, state)
VALUES ('DEMO-0001', 'History Demo', 'demo@example.com', 'CA');

In [0]:
%sql
-- Version N+2: delete the row we just added
DELETE FROM customers WHERE customer_id = 'DEMO-0001';

In [0]:
%sql
DESCRIBE HISTORY customers;

Now the history is meaningful. You should see six versions (0 through 5):

- **Version 0** -- `CREATE OR REPLACE TABLE AS SELECT`: the setup script created the table.
- **Versions 1-3** -- these three came from the single `CLUSTER BY AUTO` statement you ran in Section 3. Liquid clustering requires **row tracking** (stable IDs on every row so the engine can track which rows moved between files during re-clustering). Enabling it triggers three log entries: an **UPGRADE PROTOCOL** to a Delta version that supports row tracking, a **ROW TRACKING BACKFILL** that stamps existing rows with tracking IDs, and the **CLUSTER BY** metadata change itself.
- **Version 4** -- `WRITE`: the INSERT you just ran.
- **Version 5** -- `DELETE`: the DELETE that removed it.

Each row shows the **version**, **timestamp**, **operation**, and the user who made the change. This is what makes a UC managed table auditable and recoverable: you can see exactly what each version did, and roll back with `RESTORE TABLE ... TO VERSION AS OF <n>` if needed.

## 6. Explore lineage and history in Catalog Explorer

Unity Catalog automatically tracks how each table was created and what reads from it. This is **lineage**, and it's one of the most useful features for keeping a warehouse maintainable.

<div style="width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <!-- Row 1: Lineage steps -->
  <div style="font-size: 16px; font-weight: 700; color: #1B5162; margin-bottom: 8px;">See the lineage graph</div>
  <div style="display: flex; gap: 0; align-items: stretch; margin-bottom: 16px;">
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">1</span> Click Catalog
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65; line-height: 1.5;">In the left sidebar</div>
    </div>
    <div style="flex: 0 0 30px; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1.3; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">2</span> Navigate to table
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65;">Your catalog &#x2192; <strong>vintage_audio</strong> &#x2192; <strong>customers</strong></div>
    </div>
    <div style="flex: 0 0 30px; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1.3; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">3</span> Lineage tab
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65;">Click <strong>See lineage graph</strong> to see the <code style="font-size: 14px;">data</code> volume flowing into <code style="font-size: 14px;">customers</code></div>
    </div>
    <div style="flex: 0 0 30px; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #1B5162;">&#x2192;</div>
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">4</span> Expand nodes
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65;">Click <strong>+</strong> on any node to explore connections</div>
    </div>
  </div>
  <!-- Row 2: History -->
  <div style="font-size: 16px; font-weight: 700; color: #1B5162; margin-bottom: 8px;">See the version history</div>
  <div style="border: 2px solid #00A972; border-radius: 10px; overflow: hidden;">
    <div style="background: #00A972; color: #fff; padding: 8px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
      <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">5</span> Click the History tab
    </div>
    <div style="padding: 12px 14px; background: #F0FAF5; font-size: 15px; color: #3d5a65; line-height: 1.5;">
      You will see the same versions from <code style="font-size: 14px;">DESCRIBE HISTORY</code> in a browsable UI: CREATE TABLE (v0), protocol upgrade + backfill + CLUSTER BY (v1-3), WRITE (v4), DELETE (v5). Each row shows the operation, timestamp, and user. You can compare versions and restore from here.
    </div>
  </div>
</div>

## What you did in this lesson

<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">

The headline: on modern Databricks, **table tuning is largely automatic**. Predictive Optimization and automatic liquid clustering do the work the manual commands used to require.

| Capability | What it does | Your involvement |
|---|---|---|
| **Predictive Optimization** | Runs compaction, clustering, statistics, and vacuum automatically on UC managed tables | Confirm it's enabled; otherwise nothing |
| **Automatic liquid clustering** (`CLUSTER BY AUTO`) | PO picks and adapts clustering keys from query patterns; replaces partitioning and Z-order | One-time `CLUSTER BY AUTO` |
| `OPTIMIZE` | Compacts small files | Manual only when PO isn't managing the table |
| `ANALYZE` | Collects column statistics | Usually automatic (stats on write) |
| `VACUUM` | Deletes old unreferenced files | Usually automatic under PO |
| `DESCRIBE HISTORY` / History tab | Shows every version and its metrics | Read it when investigating or restoring |

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>